**Проверка данных на S2Looking**

In [ ]:
# Часть 3. Ячейка №2: проверка модели LEVIR-CD на S2Looking БЕЗ дообучения.
# Можно запускать после первой ячейки ИЛИ в новом сеансе Colab.
import os, json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from IPython.display import display
from google.colab import drive

# ---------- ПУТИ И НАСТРОЙКИ ----------
ROOT = Path('/content/drive/MyDrive/Компьютерное зрение (Кейс LEVIR-CD)')
RESULTS = ROOT / 'LEVIR_experiments_results'
S2_ROOT = ROOT / 'S2Looking'
MAX_PAIRS = None   # None = все доступные test-пары; для пробного запуска можно 20
PATCH_SIZE = 256
INFER_BATCH_SIZE = 8
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'

drive.mount('/content/drive')
meta_path = RESULTS/'best_model.json'
if not meta_path.is_file():
    raise FileNotFoundError(f'Нет {meta_path}: сначала запустите ЯЧЕЙКУ №1.')
meta=json.loads(meta_path.read_text(encoding='utf-8'))
if meta['input_size'] != PATCH_SIZE or meta['normalization'] != 'rgb_div_255':
    raise ValueError('Несовместимая предобработка: сверьте сохранённый best_model.json')

candidates=[S2_ROOT/'test',S2_ROOT/'S2Looking'/'test',S2_ROOT]
S2_TEST=next((p for p in candidates if all((p/q).is_dir() for q in ('Image1','Image2','label'))),None)
if S2_TEST is None:
    raise FileNotFoundError(
        f'Не найдены Image1, Image2, label в {S2_ROOT}/test. Измените S2_ROOT.'
    )
OUTPUT=RESULTS/'S2Looking_external_test'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Внешний датасет:',S2_TEST,'\nМодель:',meta['experiment']['id'])

# ---------- ТЕ ЖЕ АРХИТЕКТУРЫ, ЧТО И ПРИ ОБУЧЕНИИ ----------
class BaselineSiamese(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(True),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2))
        self.decoder=nn.Sequential(
            nn.ConvTranspose2d(128,64,2,stride=2),nn.ReLU(True),
            nn.Conv2d(64,32,3,padding=1),nn.ReLU(True),nn.Conv2d(32,1,1))
    def forward(self,a,b):
        return self.decoder(torch.cat([self.encoder(a),self.encoder(b)],dim=1))

class ConvBlock(nn.Module):
    def __init__(self,inp,out):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(inp,out,3,padding=1,bias=False),nn.BatchNorm2d(out),nn.ReLU(True),
            nn.Conv2d(out,out,3,padding=1,bias=False),nn.BatchNorm2d(out),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class SiameseUNet(nn.Module):
    def __init__(self,fusion='concat',base_channels=16):
        super().__init__()
        self.fusion=fusion
        c=base_channels
        factor=2 if fusion=='concat' else 1
        self.enc1,self.enc2,self.enc3=ConvBlock(3,c),ConvBlock(c,2*c),ConvBlock(2*c,4*c)
        self.pool=nn.MaxPool2d(2)
        self.up2=nn.ConvTranspose2d(factor*4*c,2*c,2,stride=2)
        self.dec2=ConvBlock(2*c+factor*2*c,2*c)
        self.up1=nn.ConvTranspose2d(2*c,c,2,stride=2)
        self.dec1=ConvBlock(c+factor*c,c)
        self.head=nn.Conv2d(c,1,1)
    def encode(self,x):
        f1=self.enc1(x);f2=self.enc2(self.pool(f1));f3=self.enc3(self.pool(f2))
        return f1,f2,f3
    def fuse(self,x,y):
        return torch.cat([x,y],dim=1) if self.fusion=='concat' else torch.abs(x-y)
    def forward(self,a,b):
        a1,a2,a3=self.encode(a);b1,b2,b3=self.encode(b)
        x=self.dec2(torch.cat([self.up2(self.fuse(a3,b3)),self.fuse(a2,b2)],dim=1))
        x=self.dec1(torch.cat([self.up1(x),self.fuse(a1,b1)],dim=1))
        return self.head(x)

cfg=meta['experiment']
model=(BaselineSiamese() if cfg['architecture']=='baseline' else SiameseUNet(cfg['fusion'])).to(DEVICE)
ckpt=torch.load(RESULTS/meta['checkpoint'],map_location=DEVICE,weights_only=True)
model.load_state_dict(ckpt['model_state'])
model.eval()  # только inference, НЕ optimizer / backward / обучение
THRESHOLD=float(meta['threshold'])
print(f'Загружены веса. Порог, выбранный на LEVIR-CD Validation: {THRESHOLD}')

# ---------- СПИСОК ПАР ----------
EXTS={'.png','.jpg','.jpeg','.tif','.tiff'}
files=sorted(p.name for p in (S2_TEST/'Image1').iterdir() if p.suffix.lower() in EXTS)
for f in files:
    if not (S2_TEST/'Image2'/f).is_file() or not (S2_TEST/'label'/f).is_file():
        raise FileNotFoundError(f'Отсутствует Image2 или label для {f}')
if not files: raise ValueError('В Image1 нет снимков')
if MAX_PAIRS is not None:
    if not 1<=MAX_PAIRS<=len(files): raise ValueError('Недопустимый MAX_PAIRS')
    rng=np.random.default_rng(42)
    files=sorted(rng.choice(files,size=MAX_PAIRS,replace=False).tolist())
print('Пар для независимой проверки:',len(files))

# ---------- ПРЕДСКАЗАНИЕ ПО ПАТЧАМ 256x256 БЕЗ ФИЛЬТРАЦИИ ----------
@torch.inference_mode()
def predict_large(a,b):
    h,w=a.shape[:2]
    dh=(-h)%PATCH_SIZE;dw=(-w)%PATCH_SIZE
    a=cv2.copyMakeBorder(a,0,dh,0,dw,cv2.BORDER_REFLECT_101)
    b=cv2.copyMakeBorder(b,0,dh,0,dw,cv2.BORDER_REFLECT_101)
    hh,ww=a.shape[:2]
    coords=[(y,x) for y in range(0,hh,PATCH_SIZE) for x in range(0,ww,PATCH_SIZE)]
    out=np.zeros((hh,ww),dtype=np.uint8)
    for start in range(0,len(coords),INFER_BATCH_SIZE):
        chunk=coords[start:start+INFER_BATCH_SIZE]
        aa=np.stack([a[y:y+PATCH_SIZE,x:x+PATCH_SIZE] for y,x in chunk])
        bb=np.stack([b[y:y+PATCH_SIZE,x:x+PATCH_SIZE] for y,x in chunk])
        ta=torch.from_numpy(np.ascontiguousarray(aa.transpose(0,3,1,2))).float().to(DEVICE)/255.0
        tb=torch.from_numpy(np.ascontiguousarray(bb.transpose(0,3,1,2))).float().to(DEVICE)/255.0
        with torch.autocast(device_type='cuda',enabled=AMP):
            prob=torch.sigmoid(model(ta,tb).float())[:,0]
        patches=(prob>=THRESHOLD).byte().cpu().numpy()
        for (y,x),p in zip(chunk,patches):
            out[y:y+PATCH_SIZE,x:x+PATCH_SIZE]=p
    return out[:h,:w].astype(bool)

def metrics(c):
    tp,fp,fn,tn=[int(x) for x in c]
    def div(a,b): return a/b if b else 0.0
    return {'precision':div(tp,tp+fp),'recall':div(tp,tp+fn),
            'f1':div(2*tp,2*tp+fp+fn),'iou':div(tp,tp+fp+fn),
            'accuracy':div(tp+tn,tp+fp+fn+tn),'TP':tp,'FP':fp,'FN':fn,'TN':tn}

def counts(pred,truth):
    return np.array([
        np.count_nonzero(pred & truth),np.count_nonzero(pred & ~truth),
        np.count_nonzero(~pred & truth),np.count_nonzero(~pred & ~truth)
    ],dtype=np.int64)

def load_pair(fname):
    a=cv2.imread(str(S2_TEST/'Image1'/fname),cv2.IMREAD_COLOR)
    b=cv2.imread(str(S2_TEST/'Image2'/fname),cv2.IMREAD_COLOR)
    label=cv2.imread(str(S2_TEST/'label'/fname),cv2.IMREAD_GRAYSCALE)
    if a is None or b is None or label is None:
        raise ValueError(f'Не удалось прочесть {fname}')
    if a.shape[:2]!=b.shape[:2] or a.shape[:2]!=label.shape:
        raise ValueError(f'Не совпадают размеры снимков и разметки: {fname}')
    a=cv2.cvtColor(a,cv2.COLOR_BGR2RGB)
    b=cv2.cvtColor(b,cv2.COLOR_BGR2RGB)
    # Допускаем маски как 0/255, так и 0/1.
    gt=(label>0) if label.max()<=1 else (label>127)
    return a,b,gt

# ---------- ВНЕШНЯЯ ОЦЕНКА НА ВСЕХ ПИКСЕЛЯХ ----------
total=np.zeros(4,dtype=np.int64)
matched=np.zeros(4,dtype=np.int64)  # Дополнительно: патчи с >=15 положительных пикселей
matched_patches=0
per_image=[]
for fname in tqdm(files,desc='S2Looking Test'):
    a,b,gt=load_pair(fname)
    pred=predict_large(a,b)
    c=counts(pred,gt)
    total+=c
    per_image.append({'file':fname,'changed_pixels':int(gt.sum()),**metrics(c)})
    # Дополнительный анализ с тем же правилом отбора, что при нарезке LEVIR-CD.
    h,w=gt.shape
    for y in range(0,h,PATCH_SIZE):
        for x in range(0,w,PATCH_SIZE):
            g=gt[y:y+PATCH_SIZE,x:x+PATCH_SIZE]
            p=pred[y:y+PATCH_SIZE,x:x+PATCH_SIZE]
            if int(g.sum())>=15:
                matched+=counts(p,g)
                matched_patches+=1

external=metrics(total)
external_matched=metrics(matched)
pd.DataFrame(per_image).to_csv(OUTPUT/'per_image_metrics.csv',index=False)
(OUTPUT/'external_metrics.json').write_text(json.dumps({
    'dataset':'S2Looking Test','num_image_pairs':len(files),
    'checkpoint':str(RESULTS/meta['checkpoint']),
    'threshold_from_levir':THRESHOLD,
    'all_pixels':external,
    'matched_positive_patches':{'patches':matched_patches,'metrics':external_matched}
},ensure_ascii=False,indent=2),encoding='utf-8')

lev=json.loads((RESULTS/'levir_test_metrics.json').read_text(encoding='utf-8'))['metrics']
cols=['precision','recall','f1','iou','accuracy']
comparison=pd.DataFrame([
    {'Датасет':'LEVIR-CD Test (отфильтрованные патчи)',**{k:lev[k] for k in cols}},
    {'Датасет':f'S2Looking Test (все {len(files)} пар)',**{k:external[k] for k in cols}},
    {'Датасет':f'S2Looking Test (патчи >=15 пикс. изменений; {matched_patches} патчей)',
     **{k:external_matched[k] for k in cols}}
])
comparison.to_csv(OUTPUT/'cross_dataset_comparison.csv',index=False)
print('\nСРАВНЕНИЕ РЕЗУЛЬТАТОВ:')
display(comparison.round(4))
print(f'Изменение F1 (S2Looking все пиксели − LEVIR): {(external["f1"]-lev["f1"])*100:+.2f} п.п.')
print('Важно: LEVIR-CD Test был предварительно отфильтрован, поэтому доля фона различается.')

# ---------- ЛУЧШИЕ И ХУДШИЕ ПРИМЕРЫ С НАСТОЯЩИМИ ИЗМЕНЕНИЯМИ ----------
df=pd.DataFrame(per_image)
pos=df[df.changed_pixels>0]
if not pos.empty:
    chosen=pd.concat([pos.nlargest(2,'f1'),pos.nsmallest(2,'f1')]).drop_duplicates('file')
    fig,axes=plt.subplots(len(chosen),5,figsize=(16,3.4*len(chosen)),squeeze=False)
    for i,row in enumerate(chosen.itertuples()):
        a,b,gt=load_pair(row.file)
        pred=predict_large(a,b)
        err=np.zeros((*gt.shape,3),dtype=np.uint8)
        err[gt&pred]=(30,190,30)       # TP зелёный
        err[~gt&pred]=(230,50,50)      # FP красный
        err[gt&~pred]=(50,100,240)     # FN синий
        for j,img in enumerate((a,b,gt,pred,err)):
            axes[i,j].imshow(img,cmap='gray' if j in (2,3) else None)
            axes[i,j].axis('off')
            if i==0: axes[i,j].set_title(('До','После','Разметка','Прогноз','TP/FP/FN')[j])
        axes[i,0].set_ylabel(f'F1={row.f1:.3f}')
    plt.tight_layout()
    plt.savefig(OUTPUT/'external_examples.png',dpi=150)
    plt.show()
print('\nВнешняя проверка завершена. Файлы сохранены:',OUTPUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Внешний датасет: /content/drive/MyDrive/Компьютерное зрение (Кейс LEVIR-CD)/S2Looking 
Модель: E3
Загружены веса. Порог, выбранный на LEVIR-CD Validation: 0.7
Пар для независимой проверки: 1000


S2Looking Test:   0%|          | 0/1000 [00:00<?, ?it/s]